In [1]:
import pandas as pd

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

In [21]:
from string import punctuation

In [26]:
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
df = pd.read_csv("d:/git/dados/nlp/news_sentiment_analysis.csv", encoding="utf-8")

In [7]:
df = df.drop(columns=["Source", "Author", "URL", "Published At"])

In [10]:
df

,Title,Description,Sentiment,Type
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,Business
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",neutral,Business
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,positive,Business
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,negative,Business
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,positive,Business
...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,positive,Technology
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,positive,Technology
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,positive,Technology
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology


In [11]:
mapping = {"positive": 1, "negative": 0, "neutral": 0}
df["sentiment_number"] = df["Sentiment"].map( mapping ) 

In [12]:
df

,Title,Description,Sentiment,Type,sentiment_number
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,Business,1
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",neutral,Business,0
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,positive,Business,1
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,negative,Business,0
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,positive,Business,1
...,...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,positive,Technology,1
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,positive,Technology,1
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,positive,Technology,1
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology,0


In [55]:
Y = torch.tensor(df["sentiment_number"], dtype=torch.float32)
Y = torch.reshape(Y, (-1, 1))

In [56]:
Y.shape

torch.Size([3500, 1])

In [22]:
table = str.maketrans("", "", punctuation)

def limpar_titulo( texto ):
    texto_limpo = texto.lower().translate(table)
    return texto_limpo
    

In [24]:
df["title_clean"] = df["Title"].apply(limpar_titulo)

In [87]:
df[["title_clean", "sentiment_number"]][0:10]

,title_clean,sentiment_number
0,pine view high teacher wins best in state awar...,1
1,businesses face financial strain amid liquidit...,0
2,musk donates to super pac working to elect tru...,1
3,us ftc issues warning to franchisors over unfa...,0
4,rooftop solars dark side,1
5,gabelli asks paramount for details on national...,1
6,qwi investments qwi ndash announcement re net...,1
7,rome resources announces shareholder approval ...,1
8,fawcett accused of fronting scheme for hostile...,1
9,what makes spynn publicity the top choice for ...,1


In [34]:
vetorizador = CountVectorizer(max_features=1000)

In [35]:
texto_vetorizado = vetorizador.fit_transform(df["title_clean"])

In [72]:
dicionario = vetorizador.get_feature_names_out()
dicionario

array(['038', '10', '100', '11', '12', '13', '14', '15', '16', '17', '18',
       '18th', '20', '2023', '2024', '202425', '2025', '2028', '2031',
       '20th', '24', '25', '26', '38', '3rd', '647th', '8th', 'about',
       'access', 'acquired', 'acquires', 'acquisition', 'across', 'act',
       'action', 'ad', 'advisers', 'advisors', 'advisory', 'afb', 'after',
       'against', 'age', 'agency', 'agreement', 'ahead', 'ai', 'aid',
       'air', 'aircraft', 'aktie', 'al', 'aldi', 'all', 'allelectric',
       'allstar', 'alto', 'amazon', 'america', 'american', 'americas',
       'amid', 'amkor', 'among', 'an', 'analysis', 'analysts',
       'analytics', 'and', 'anniversary', 'announce', 'announces',
       'annual', 'another', 'antitrans', 'ap', 'app', 'apple',
       'applications', 'appointment', 'approach', 'are', 'argentina',
       'armed', 'army', 'art', 'artillery', 'arts', 'as', 'asia', 'asks',
       'asset', 'astronaut', 'at', 'attack', 'attorney', 'attractions',
       'auf', 

In [50]:
X = torch.tensor(texto_vetorizado.toarray(), dtype=torch.float32)
X

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [66]:
print("X: ", X.dtype, X.shape, X.ndim)
print("Y: ", Y.dtype, Y.shape, Y.ndim)

X:  torch.float32 torch.Size([3500, 1000]) 2
Y:  torch.float32 torch.Size([3500, 1]) 2


In [67]:
modelo = nn.Sequential(
    nn.Linear(in_features = 1000, out_features=1),
    nn.Sigmoid()
)

In [68]:
criterio = nn.BCELoss() # Binary Cross Entropy
otimizador = optim.SGD( modelo.parameters(), lr=1 )

In [70]:
for epoca in range(1, 3001):
    Y_hat = modelo( X )
    loss = criterio( Y_hat, Y )
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    print(f"Epoca: {epoca}\tLoss:{loss}")

Epoca: 1	Loss:0.4037390351295471
Epoca: 2	Loss:0.4036838710308075
Epoca: 3	Loss:0.40362873673439026
Epoca: 4	Loss:0.4035736322402954
Epoca: 5	Loss:0.4035186469554901
Epoca: 6	Loss:0.4034636914730072
Epoca: 7	Loss:0.40340879559516907
Epoca: 8	Loss:0.4033539295196533
Epoca: 9	Loss:0.4032991826534271
Epoca: 10	Loss:0.4032445549964905
Epoca: 11	Loss:0.40318986773490906
Epoca: 12	Loss:0.4031353294849396
Epoca: 13	Loss:0.4030807614326477
Epoca: 14	Loss:0.403026282787323
Epoca: 15	Loss:0.40297189354896545
Epoca: 16	Loss:0.4029175639152527
Epoca: 17	Loss:0.4028632342815399
Epoca: 18	Loss:0.4028090536594391
Epoca: 19	Loss:0.40275490283966064
Epoca: 20	Loss:0.4027007818222046
Epoca: 21	Loss:0.4026467502117157
Epoca: 22	Loss:0.4025927782058716
Epoca: 23	Loss:0.40253889560699463
Epoca: 24	Loss:0.4024850130081177
Epoca: 25	Loss:0.4024312198162079
Epoca: 26	Loss:0.40237751603126526
Epoca: 27	Loss:0.40232381224632263
Epoca: 28	Loss:0.4022701680660248
Epoca: 29	Loss:0.4022165834903717
Epoca: 30	Loss:0

In [73]:
vetorizador_predict = CountVectorizer(max_features=1000, vocabulary=dicionario)

In [84]:
X_predict = vetorizador_predict.fit_transform( df["title_clean"][0:10] )
X_predict.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(10, 1000))

In [86]:
with torch.no_grad():
    X_pred = torch.tensor(X_predict.toarray(), dtype=torch.float32)
    resposta = modelo( X_pred )
    print(f"Resposta: {resposta}")

Resposta: tensor([[0.9879],
        [0.3327],
        [0.8803],
        [0.6030],
        [0.6939],
        [0.8062],
        [0.3120],
        [0.7129],
        [0.7087],
        [0.8809]])
